In [17]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/raw/retail_store_sales.csv")

print("First Look & Sample Data:")
print(df.head())
print("\n")
print("Data Information:")
print(df.info())
print("\n")
print("Data Shape:")
print(df.shape)
print("\n")
print("Summary Statistics:")
print(df.describe(include='all'))


First Look & Sample Data:
  Transaction ID Customer ID       Category          Item  Price Per Unit  \
0    TXN_6867343     CUST_09     Patisserie   Item_10_PAT            18.5   
1    TXN_3731986     CUST_22  Milk Products  Item_17_MILK            29.0   
2    TXN_9303719     CUST_02       Butchers   Item_12_BUT            21.5   
3    TXN_9458126     CUST_06      Beverages   Item_16_BEV            27.5   
4    TXN_4575373     CUST_05           Food   Item_6_FOOD            12.5   

   Quantity  Total Spent  Payment Method Location Transaction Date  \
0      10.0        185.0  Digital Wallet   Online       2024-04-08   
1       9.0        261.0  Digital Wallet   Online       2023-07-23   
2       2.0         43.0     Credit Card   Online       2022-10-05   
3       9.0        247.5     Credit Card   Online       2022-05-07   
4       7.0         87.5  Digital Wallet   Online       2022-10-02   

  Discount Applied  
0             True  
1             True  
2            False  
3     

In [18]:
print("Missing Values:")
print(df.isna().sum())
print("\n")
print("Percentage of missing values:")
print((df.isna().mean() * 100).round(1))

Missing Values:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


Percentage of missing values:
Transaction ID       0.0
Customer ID          0.0
Category             0.0
Item                 9.6
Price Per Unit       4.8
Quantity             4.8
Total Spent          4.8
Payment Method       0.0
Location             0.0
Transaction Date     0.0
Discount Applied    33.4
dtype: float64


In [19]:
print("Other missings:")
for col in ["Category", "Item", "Payment Method", "Location", "Discount Applied"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

Other missings:
--- Category ---
Category
Furniture                             1591
Electric household essentials         1591
Food                                  1588
Milk Products                         1584
Butchers                              1568
Beverages                             1567
Computers and electric accessories    1558
Patisserie                            1528
Name: count, dtype: int64

--- Item ---
Item
NaN            1213
Item_2_BEV      126
Item_25_FUR     113
Item_11_FUR     110
Item_1_MILK     109
               ... 
Item_5_BEV        7
Item_13_FUR       7
Item_24_CEA       7
Item_21_PAT       6
Item_3_EHE        5
Name: count, Length: 201, dtype: int64

--- Payment Method ---
Payment Method
Cash              4310
Digital Wallet    4144
Credit Card       4121
Name: count, dtype: int64

--- Location ---
Location
Online      6354
In-store    6221
Name: count, dtype: int64

--- Discount Applied ---
Discount Applied
True     4219
NaN      4199
False    4157
Name

In [20]:
print("Duplicates:")
print(df.duplicated().sum())
print("\n")
print("Duplicates for unique columns:")
print(df["Transaction ID"].duplicated().sum())

Duplicates:
0


Duplicates for unique columns:
0


In [21]:
print("TypeErrors:")
print(df.dtypes)
print("\n")
print("Numeric errors")
print("\n")
print("Quantity column errors:")
print(pd.to_numeric(df["Quantity"], errors='coerce').isna().sum())
print("\n")
print("Price column errors:")
print(pd.to_numeric(df["Price Per Unit"], errors='coerce').isna().sum())
print("\n")
print("Total spent column errors:")
print(pd.to_numeric(df["Total Spent"], errors='coerce').isna().sum())
print("\n")

print("Date column errors:")
print(pd.to_datetime(df["Transaction Date"], errors='coerce').isna().sum())




TypeErrors:
Transaction ID       object
Customer ID          object
Category             object
Item                 object
Price Per Unit      float64
Quantity            float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
Discount Applied     object
dtype: object


Numeric errors


Quantity column errors:
604


Price column errors:
609


Total spent column errors:
604


Date column errors:
0


In [22]:
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]
for col in numeric_columns:
    q1=df[col].quantile(0.25)
    q3=df[col].quantile(0.75)
    iqr=q3-q1

    low=q1-1.5*iqr
    high=q3+1.5*iqr

    outliers=df[(df[col]<low) | (df[col]>high)]
    print(f"---Column: {col}---")
    print(f"Limits: {low} - {high}")
    print(f"Outliers: {len(outliers)}")

---Column: Quantity---
Limits: -4.5 - 15.5
Outliers: 0
---Column: Price Per Unit---
Limits: -15.25 - 62.75
Outliers: 0
---Column: Total Spent---
Limits: -160.5 - 403.5
Outliers: 60


In [23]:
df_clean=df.copy()

In [24]:
log = []

df_clean["Transaction Date"]=pd.to_datetime(df_clean["Transaction Date"] , errors='coerce')
df_clean["Discount Applied"]=df_clean["Discount Applied"].astype("boolean")
df_clean["Quantity"]=pd.to_numeric(df_clean["Quantity"], errors='coerce')

log.append("Converted 'Transaction Date' to datetime, 'Discount Applied' to boolean, and 'Quantity' to numeric.")   

print(df_clean.dtypes)
print("\n")
print(log)




Transaction ID              object
Customer ID                 object
Category                    object
Item                        object
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Discount Applied           boolean
dtype: object


["Converted 'Transaction Date' to datetime, 'Discount Applied' to boolean, and 'Quantity' to numeric."]


In [25]:


q, p , t ="Quantity", "Price Per Unit", "Total Spent"
print(f"Missing values before cleaning: " )
print(df_clean[[q,p,t]].isna().sum())

m_=df_clean[t].isna() & df_clean[q].notna() & df_clean[p].notna()
print("Lines: " + str(m_.sum()))
df_clean.loc[m_, t]=df_clean.loc[m_, q]*df_clean.loc[m_, p]
log.append(f"Filled missing values in '{t}' using '{q}' and '{p}'.")

m=df_clean[q].isna() & df_clean[t].notna() & df_clean[p].notna()
print("Lines: " + str(m.sum()))
df_clean.loc[m, q]=df_clean.loc[m, t]/df_clean.loc[m, p]
log.append(f"Filled missing values in '{q}' using '{t}' and '{p}'.")

m=df_clean[p].isna() & df_clean[t].notna() & df_clean[q].notna()
print("Lines: " + str(m.sum()))
df_clean.loc[m, p]=df_clean.loc[m, t]/df_clean.loc[m, q]
log.append(f"Filled missing values in '{p}' using '{t}' and '{q}'.")



Missing values before cleaning: 
Quantity          604
Price Per Unit    609
Total Spent       604
dtype: int64
Lines: 0
Lines: 0
Lines: 609


In [26]:

print("Missing values after cleaning: " )
print(df_clean[[q,p,t]].isna().sum())
print("\n")
print("Infinity Values: ")
print(np.isinf(df_clean[[q,p,t]]).sum())

Missing values after cleaning: 
Quantity          604
Price Per Unit      0
Total Spent       604
dtype: int64


Infinity Values: 
Quantity          0
Price Per Unit    0
Total Spent       0
dtype: int64


In [ ]:
rows_before=len(df_clean)
df_clean=df_clean.dropna(subset=[q,t])

rows_dropped=rows_before-len(df_clean)
log.append(f"Dropped {rows_dropped} unrecoverable rows ")

print(f"Succesfully Deleted: {rows_dropped} rows ")

print("\n")

print("After dropna :")
print(df_clean[[q,p,t]].isna().sum())

Succesful Deleted: 604 rows 


After dropna :
Quantity          0
Price Per Unit    0
Total Spent       0
dtype: int64


In [28]:
print(df_clean.isna().sum())

Transaction ID         0
Customer ID            0
Category               0
Item                 609
Price Per Unit         0
Quantity               0
Total Spent            0
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    3988
dtype: int64


In [29]:
df_clean["Discount Applied"]=df_clean["Discount Applied"].fillna(False)
log.append("Filled missing values in 'Discount Applied' with False")



In [ ]:
df_clean["Item"]=df_clean["Item"].fillna("Unknown Item")
log.append("Filled missing values in 'Item' with Unknown Item")


In [31]:
print(df_clean.isna().sum())

Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64


In [33]:
invalid_q=(df_clean[q]<=0).sum()
invalid_p=(df_clean[p]<=0).sum()
invalid_t=(df_clean[t]<=0).sum()
print(f"Quantity <= 0: {invalid_q}")
print(f"Price Per Unit <= 0: {invalid_p}")
print(f"Total Spent <= 0: {invalid_t}")

discrepancies=(
    np.abs(df_clean[t]-(df_clean[p]*df_clean[q]))>0.01
).sum()

print(discrepancies)

duplicates=df_clean.duplicated().sum()
print(f"duplicates  {duplicates}")


Quantity <= 0: 0
Price Per Unit <= 0: 0
Total Spent <= 0: 0
0
duplicates  0


In [34]:
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

print("=== OUTLIER CHECK (AFTER CLEANING) ===")
for col in numeric_columns:
  q1 = df_clean[col].quantile(0.25)
  q3 = df_clean[col].quantile(0.75)
  iqr = q3 - q1

  low = q1 - 1.5 * iqr
  high = q3 + 1.5 * iqr

  outliers = df_clean[(df_clean[col] < low) | (df_clean[col] > high)]

  print(f"--- Column: {col} ---")
  print(f"Limits: [{low:.2f}, {high:.2f}]")
  print(f"Outliers: {len(outliers)}")

=== OUTLIER CHECK (AFTER CLEANING) ===
--- Column: Quantity ---
Limits: [-4.50, 15.50]
Outliers: 0
--- Column: Price Per Unit ---
Limits: [-15.25, 62.75]
Outliers: 0
--- Column: Total Spent ---
Limits: [-160.50, 403.50]
Outliers: 60


In [35]:
print(df_clean[numeric_columns].max())

Quantity           10.0
Price Per Unit     41.0
Total Spent       410.0
dtype: float64


In [38]:
log.append("Total Spent: 60 outliers (IQR, over 403.5) kept; max 410 = 10 x 41 (max Quantity x max Price), so they are genuine large purchases, not errors.")

In [39]:
from pathlib import Path
Path("../data/clean").mkdir(parents=True, exist_ok=True)
Path("../reports").mkdir(parents=True, exist_ok=True)

df_clean.to_csv("../data/clean/retail_store_sales_clean.csv", index=False)
pd.DataFrame({"step": range(1, len(log) + 1), "action": log}).to_csv("../reports/cleaning_log.csv", index=False)
print("Saved:", len(df_clean), "rows,", len(log), "log entries")

Saved: 11971 rows, 10 log entries


In [40]:
print("Rows before/after:", len(df), "/", len(df_clean))
nulls = pd.DataFrame({"nulls_before": df.isna().sum(), "nulls_after": df_clean.isna().sum()})
print(nulls)

cols = [q, p, t]
stats = pd.DataFrame({
       "mean_before": df[cols].mean(), "mean_after": df_clean[cols].mean(),
       "median_before": df[cols].median(), "median_after": df_clean[cols].median(),
       "std_before": df[cols].std(), "std_after": df_clean[cols].std(),
   }).round(2)
print(stats)

nulls.join(stats, how="outer").to_csv("../reports/before_after_comparison.csv")

Rows before/after: 12575 / 11971
                  nulls_before  nulls_after
Transaction ID               0            0
Customer ID                  0            0
Category                     0            0
Item                      1213            0
Price Per Unit             609            0
Quantity                   604            0
Total Spent                604            0
Payment Method               0            0
Location                     0            0
Transaction Date             0            0
Discount Applied          4199            0
                mean_before  mean_after  median_before  median_after  \
Quantity               5.54        5.54            6.0           6.0   
Price Per Unit        23.37       23.36           23.0          23.0   
Total Spent          129.65      129.65          108.5         108.5   

                std_before  std_after  
Quantity              2.86       2.86  
Price Per Unit       10.74      10.74  
Total Spent          94.75    